In [1]:
from pathlib import Path
import re

___
### Extracting useful info from log file

Log file contains lot of metadata, such as date, time, process id at the beginning of each line. This is unnecessary for our data<br><br>
We only need lines with block id information, and the message in the line to build template<br><br>
Example line from the log:<br>
081111 041138 19464 INFO dfs.DataNodeDataXceiver: Receiving block blk_-4750397141673817648 src: /10.251.70.112:52693 dest: /10.251.70.112:50010<br><br>
Here, "081111 041138 19464 INFO dfs.DataNodeDataXceiver" is unnecessary. We only need "blk_-4750397141673817648", and "Receiving block blk_-4750397141673817648 src:(.) dest: (.) <br><br>

Regex for block id: (blk_-?\d+)<br>
&ensp;blk_  -  must be present<br>
&ensp;-?    -  '-' is optional<br>
&ensp;\d+   -   any number of digits after that is acceptable<br><br>

Parsing message:<br>
From sample line, we understand that message part lies after the first colon. Hence, split line based on first colon, then second part of sentence will be message. Message still not cleaned properly at this stage, as things like src and dest will still have numbers

In [4]:
log_path = Path('../datasets/HDFS_v1/HDFS.log')

def find_pattern(log_path): 
    data = []
    
    blk_pattern = re.compile(r'(blk_-?\d+)')

    with open(log_path) as f:
        for line in f:
            blk_match = blk_pattern.search(line)
            if not blk_match:
                continue

            blk_id = blk_match.group(1)

            parts = line.split(':', maxsplit=1)
            if len(parts) < 2:
                continue
            msg = parts[1].strip()

            data.append((blk_id, msg))

    return data

In [9]:
parsed_data = find_pattern(log_path)
print('No of lines parsed: ', len(parsed_data))

No of lines parsed:  11175629


In [8]:
print('15 sample lines: ')
for i in range(15):
    print(parsed_data[i])

15 sample lines: 
('blk_-1608999687919862906', 'Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010')
('blk_-1608999687919862906', 'BLOCK* NameSystem.allocateBlock: /mnt/hadoop/mapred/system/job_200811092030_0001/job.jar. blk_-1608999687919862906')
('blk_-1608999687919862906', 'Receiving block blk_-1608999687919862906 src: /10.250.10.6:40524 dest: /10.250.10.6:50010')
('blk_-1608999687919862906', 'Receiving block blk_-1608999687919862906 src: /10.250.14.224:42420 dest: /10.250.14.224:50010')
('blk_-1608999687919862906', 'PacketResponder 1 for block blk_-1608999687919862906 terminating')
('blk_-1608999687919862906', 'PacketResponder 2 for block blk_-1608999687919862906 terminating')
('blk_-1608999687919862906', 'Received block blk_-1608999687919862906 of size 91178 from /10.250.10.6')
('blk_-1608999687919862906', 'Received block blk_-1608999687919862906 of size 91178 from /10.250.19.102')
('blk_-1608999687919862906', 'PacketResponder 0 for block